# Robustness checks: does exposure really bring abstract and representational art closer?

**Why this notebook.** In the main DINOv2 run, training by exposure (no labels) reduced the distance between the average abstract and the average representational painting by about 24% in all 3 seeds, while labels pushed them apart. In DINOv3 exposure did *not* do this. Before interpreting the DINOv2 result, this notebook checks whether it is genuine or an artefact of how we measured it.

**The checks**
1. **Measures that don't rely on group averages.** The original "between" distance compares two averages, which can shift for reasons unrelated to the paintings themselves. Here we add:
   - **pairwise distance ratio:** average distance between abstract–representational *pairs* of paintings ÷ average distance between pairs from the same group (higher = more separate);
   - **neighbour mixing:** how often a painting's 10 nearest neighbours come from the *other* group, relative to chance (0 = groups fully separate, 1 = fully mixed);
   - **held-out separability (d′):** how well a linear boundary learned on some paintings separates the groups for unseen paintings (higher = more separate).
2. **Alternative group definitions:** Cubism counted as abstract, Pop Art as representational, single styles left out, and so on. A real effect shouldn't hinge on one choice.
3. **Control splits unrelated to abstraction** (for example Baroque + Realism vs Impressionism + Ukiyo-e). If exposure brings *these* closer too, it is merging everything, not specifically bridging abstract and representational art.
4. **Style-by-style map:** how the distance between every pair of the 8 styles changed under labels and under exposure.

**The result counts as robust if** all three measures agree in every seed, it holds for most alternative groupings, and abstract–representational pairs move closer more than control pairs.

**Setup.** Retrains DINOv2 with the same settings as the main run (warm-up 3, learning rate 3e-5, last 4 blocks, 30 epochs, 3 seeds, gradient clipping): about 30 minutes on a T4. **All embeddings are saved to a file**, so later you can set `LOAD_PATH` and skip training. To check DINOv3 exposure the same way, add its run to `RUNS` (with its stronger settings).

In [ ]:
!pip -q install datasets umap-learn "transformers>=4.56"

In [ ]:
import os
os.environ["USE_TF"] = "0"

import copy, gc, math, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms as T
from datasets import load_dataset
from transformers import AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import umap

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU attached. Switch to a T4 GPU runtime.")

**Kaggle only:** uncomment to load your Hugging Face token (needed only for DINOv3).

In [ ]:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Settings

In [ ]:
RUNS = [   # one entry per model + objective; lr and n_blocks can differ per run
    dict(model="facebook/dinov2-small", objective="labels", lr=3e-5, n_blocks=4),
    dict(model="facebook/dinov2-small", objective="exposure", lr=3e-5, n_blocks=4),
    # dict(model="facebook/dinov3-vits16-pretrain-lvd1689m", objective="exposure", lr=1e-4, n_blocks=8),
]
MODELS = list(dict.fromkeys(r["model"] for r in RUNS))
OBJECTIVES = list(dict.fromkeys(r["objective"] for r in RUNS))
SAVE_PATH = "robustness_embeddings.npz"   # embeddings are written here after training
LOAD_PATH = None                          # set to the saved file to skip training, e.g. "robustness_embeddings.npz"
WARMUPS = [3]             # head warm-up epochs (model frozen during warm-up)

LR, N_BLOCKS = 3e-5, 4    # defaults; each run in RUNS sets its own
EPOCHS = 30
RECORD_EVERY = 2          # after the dense early phase, measure every N epochs
DENSE_UNTIL = 6           # measure every epoch up to here
BATCH = {"labels": 32, "exposure": 64}
SSL_TEMPERATURE = 0.1
USE_AMP = True            # mixed precision: ~2x faster on a T4
AMP_OFF_FOR = ["dinov3"]  # models that need full precision (DINOv3 overflows 16-bit numbers → NaN loss)
GRAD_CLIP = 1.0           # limit sudden large updates (None to switch off)
SEEDS = [0, 1, 2]
K_NEIGHBOURS = 10

STYLES = ["Impressionism", "Baroque", "Cubism", "Ukiyo_e", "Pop_Art",
          "Abstract_Expressionism", "Realism", "Color_Field_Painting"]
PER_STYLE = 120
MAX_SCAN = 25000
ABSTRACT = ["Abstract_Expressionism", "Color_Field_Painting"]
REPRESENTATIONAL = ["Baroque", "Realism", "Impressionism", "Ukiyo_e"]

QUERY = (0.35, 0.5)       # spatial maps: query patch position as (row, column) fractions of the image
RECORD_EPOCHS = sorted(set([0] + list(range(1, min(DENSE_UNTIL, EPOCHS) + 1))
                           + list(range(RECORD_EVERY, EPOCHS + 1, RECORD_EVERY)) + [EPOCHS]))
print("Recording at epochs:", RECORD_EPOCHS)

## Load paintings

Same paintings and splits as the main run.

In [ ]:
torch.manual_seed(0); np.random.seed(0); random.seed(0)
ds = load_dataset("huggan/wikiart", split="train", streaming=True)
if ds.features is None:
    ds = ds._resolve_features()
all_styles = ds.features["style"].names
style_ids = {all_styles.index(s): s for s in STYLES if s in all_styles}
id_to_label = {sid: i for i, sid in enumerate(style_ids)}
class_names = [style_ids[sid] for sid in style_ids]
K = len(class_names)

counts = {sid: 0 for sid in style_ids}
images, labels = [], []
for i, ex in enumerate(ds.shuffle(seed=0, buffer_size=1000)):
    sid = ex["style"]
    if sid in counts and counts[sid] < PER_STYLE:
        img = ex["image"].convert("RGB")
        img.thumbnail((320, 320))
        images.append(img); labels.append(id_to_label[sid]); counts[sid] += 1
    if all(c >= PER_STYLE for c in counts.values()) or i >= MAX_SCAN:
        break
labels = np.array(labels)
del ds; gc.collect()

tr_idx, te_idx = train_test_split(np.arange(len(images)), test_size=0.2, stratify=labels, random_state=0)
fit_idx, val_idx = train_test_split(tr_idx, test_size=0.2, stratify=labels[tr_idx], random_state=0)
ia_all = np.where(np.isin(labels, [class_names.index(n) for n in ABSTRACT if n in class_names]))[0]
ir_all = np.where(np.isin(labels, [class_names.index(n) for n in REPRESENTATIONAL if n in class_names]))[0]
test_imgs = [images[j] for j in te_idx]
EX_POS = [int(np.where(labels[te_idx] == k)[0][0]) for k in range(min(3, K))]   # example paintings for the maps
print(f"Kept {len(images)} paintings: {len(fit_idx)} fit / {len(val_idx)} validation / {len(te_idx)} test")

## Helpers

In [ ]:
def short(model_id):
    return model_id.split("/")[-1].replace("-pretrain-lvd1689m", "")

def probe_on(emb, train_ix, eval_ix):
    sc = StandardScaler().fit(emb[train_ix])
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(emb[train_ix]), labels[train_ix])
    return clf.score(sc.transform(emb[eval_ix]), labels[eval_ix])

probe = lambda emb: probe_on(emb, tr_idx, te_idx)
val_probe = lambda emb: probe_on(emb, fit_idx, val_idx)

def split_parts(emb, ia, ir):
    """between: cosine distance between the abstract and representational centres;
       within: average cosine distance of paintings to their own group's centre;
       overall: average cosine distance of all paintings to the overall centre (general spread)."""
    e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    A, R_ = e[ia], e[ir]
    ca, cr = A.mean(0), R_.mean(0)
    between = 1 - ca @ cr / (np.linalg.norm(ca) * np.linalg.norm(cr))
    within = np.mean([np.mean(1 - X @ c / np.linalg.norm(c)) for X, c in ((A, ca), (R_, cr))])
    g = e.mean(0)
    overall = np.mean(1 - e @ g / np.linalg.norm(g))
    return between, within, overall

def split_ratio(emb, ia, ir):
    between, within, _ = split_parts(emb, ia, ir)
    return between / within

def split_record(emb):
    b, w, o = split_parts(emb, ia_all, ir_all)
    return dict(split_between=b, split_within=w, overall_spread=o)

def linear_cka(X, Y):
    X = X.astype(np.float64) - X.mean(0); Y = Y.astype(np.float64) - Y.mean(0)
    return np.linalg.norm(Y.T @ X) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))

def show_img(img):
    """Same framing the models see: shorter side to 256, centre crop 224."""
    w, h = img.size
    sc = 256 / min(w, h)
    img = img.resize((max(256, round(w * sc)), max(256, round(h * sc))), Image.BICUBIC)
    w, h = img.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return img.crop((l, t, l + 224, t + 224))

def upsample(a, size=224):
    a = np.asarray(Image.fromarray(a.astype(np.float32)).resize((size, size), Image.BILINEAR))
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def unit_rows(E):
    E = E.astype(np.float64)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def procrustes_rotation(B, A):
    """Rotation R (orthogonal) that best maps B onto A, fitted on centred data."""
    U, _, Vt = np.linalg.svd((B - B.mean(0)).T @ (A - A.mean(0)))
    return U @ Vt

def field_stats(E0, Et, align=True):
    """Split how paintings moved (E0 -> Et, unit-normalised) into:
       shared   = one common shift for all paintings (the steering *vector*),
       style    = extra movement shared within each style (a style-organised *field*),
       other    = movement specific to individual paintings.
    Fractions of total squared movement; they add to 1."""
    A, B = unit_rows(E0), unit_rows(Et)
    if align:
        B = B @ procrustes_rotation(B, A)       # remove any pure rotation of the space
    D = B - A
    total = (D ** 2).sum(1).mean()
    mean_shift = D.mean(0)
    shared = (mean_shift ** 2).sum()
    Dr = D - mean_shift
    style = sum((labels == k).sum() * (Dr[labels == k].mean(0) ** 2).sum() for k in range(K)) / len(labels)
    u = A[ia_all].mean(0) - A[ir_all].mean(0); u /= np.linalg.norm(u)
    proj = Dr @ u                                # + = towards the abstract side of the original axis
    return dict(total=total, shared=shared / total, style=style / total, other=1 - (shared + style) / total,
                abstract_move=proj[ia_all].mean(), representational_move=proj[ir_all].mean(), D=D, Dr=Dr, B=B, A=A)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
label_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
ssl_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.3, 1.0)), T.RandomHorizontalFlip(),
                    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
                    T.ToTensor(), T.Normalize(MEAN, STD)])
AMP = USE_AMP and device == "cuda"

def load_backbone(model_id):
    return AutoModel.from_pretrained(model_id, attn_implementation="eager").to(device).eval()

def cls_of(model, x):
    return model(pixel_values=x).last_hidden_state[:, 0]

@torch.no_grad()
def embed(model, imgs, bs=64):
    model.eval()
    return torch.cat([cls_of(model, torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)).float().cpu()
                      for i in range(0, len(imgs), bs)]).numpy()

def get_blocks(bb):
    for path in ("encoder.layer", "model.layer", "layer"):
        obj = bb
        try:
            for name in path.split("."):
                obj = getattr(obj, name)
        except AttributeError:
            continue
        if isinstance(obj, nn.ModuleList):
            return obj
    raise ValueError("Could not find the transformer blocks")

def get_final_norm(bb):
    return bb.layernorm if hasattr(bb, "layernorm") else bb.norm

def make_head(objective, hidden):
    if objective == "labels":
        return nn.Linear(hidden, K)
    return nn.Sequential(nn.Linear(hidden, 512), nn.GELU(), nn.Linear(512, 128))

def nt_xent(z, tau):
    z = F.normalize(z.float(), dim=1)            # computed in float32 for stability
    n = z.shape[0] // 2
    sim = z @ z.T / tau - 1e9 * torch.eye(2 * n, device=z.device)
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

@torch.no_grad()
def head_accuracy(model, head, index, bs=64):
    model.eval(); head.eval()
    correct = 0
    for i in range(0, len(index), bs):
        b = index[i:i + bs]
        x = torch.stack([eval_tf(images[j]) for j in b]).to(device)
        correct += int((head(cls_of(model, x)).argmax(1).cpu().numpy() == labels[b]).sum())
    return correct / len(index)

@torch.no_grad()
def trainable_change(base, model):
    pb = dict(base.named_parameters())
    d = n = 0.0
    for name, p in model.named_parameters():
        if p.requires_grad:
            d += (p - pb[name]).float().pow(2).sum().item(); n += pb[name].float().pow(2).sum().item()
    return math.sqrt(d) / (math.sqrt(n) + 1e-12)

@torch.no_grad()
def patch_sims(model, imgs, bs=32):
    """Per painting: cosine similarity between every pair of final-layer patch tokens. (images, P, P), float16 on GPU."""
    model.eval()
    out = []
    for i in range(0, len(imgs), bs):
        px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
        h = model(pixel_values=px).last_hidden_state[:, 1:]
        side = int(math.sqrt(h.shape[1]))
        h = F.normalize(h[:, -side * side:].float(), dim=-1)     # patch tokens only (drops register tokens)
        out.append((h @ h.transpose(1, 2)).half())
    return torch.cat(out)

_MASKS = {}
def grid_masks(side, dev):
    if side not in _MASKS:
        yy, xx = torch.meshgrid(torch.arange(side), torch.arange(side), indexing="ij")
        c = torch.stack([yy.flatten(), xx.flatten()], 1).float()
        d = torch.cdist(c, c)
        near = (d > 0) & (d <= 1.5)                                # the 8 neighbouring patches
        far = d >= side / 2                                        # patches at least half the image apart
        triu = torch.triu(torch.ones_like(d, dtype=torch.bool), diagonal=1)
        _MASKS[side] = (near.to(dev), far.to(dev), triu.to(dev))
    return _MASKS[side]

@torch.no_grad()
def spatial_metrics(S, S0):
    """local: neighbours minus distant similarity (higher = more spatially organised);
       homog: mean similarity of all patch pairs (higher = patches more alike, global info swamping local);
       preserve: correlation of each painting's patch-similarity pattern with the untrained model's (1 = unchanged)."""
    near, far, triu = grid_masks(int(math.sqrt(S.shape[-1])), S.device)
    S, S0 = S.float(), S0.float()
    local = S[:, near].mean(1) - S[:, far].mean(1)
    a, b = S[:, triu], S0[:, triu]
    homog = a.mean(1)
    a = a - a.mean(1, keepdim=True); b = b - b.mean(1, keepdim=True)
    preserve = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)
    return dict(spatial_local=local.mean().item(), spatial_homog=homog.mean().item(),
                spatial_preserve=preserve.mean().item())

def train_long(base, objective, seed, on_record, warmup=0):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = copy.deepcopy(base).to(device)
    for p in model.parameters():
        p.requires_grad = False
    blocks = get_blocks(model)
    for block in blocks[max(len(blocks) - N_BLOCKS, 0):]:
        for p in block.parameters():
            p.requires_grad = True
    for p in get_final_norm(model).parameters():
        p.requires_grad = True
    head = make_head(objective, model.config.hidden_size).to(device)
    bs = BATCH[objective]

    # Head warm-up: train only the new head on the frozen model, so random head weights can't disturb the features
    head_opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.05)
    for _ in range(warmup):
        model.eval(); head.train()
        order = np.random.permutation(fit_idx)
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                feats = cls_of(model, x).float()
            if objective == "labels":
                loss = F.cross_entropy(head(feats), torch.tensor(labels[b]).to(device))
            else:
                loss = nt_xent(head(feats), SSL_TEMPERATURE)
            head_opt.zero_grad(set_to_none=True); loss.backward(); head_opt.step()
    del head_opt

    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if p.requires_grad], "lr": LR},
        {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    trainable = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
    for epoch in range(1, EPOCHS + 1):
        model.train(); head.train()
        order, losses = np.random.permutation(fit_idx), []
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                    loss = F.cross_entropy(head(cls_of(model, x)).float(), torch.tensor(labels[b]).to(device))
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                    loss = nt_xent(head(cls_of(model, x)), SSL_TEMPERATURE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if GRAD_CLIP:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        if epoch in RECORD_EPOCHS:
            on_record(epoch, model, head, float(np.mean(losses)))
    del model, head, opt

## Train (or load) and save embeddings

Embeddings of all 960 paintings are kept at every recorded epoch. Training is skipped if `LOAD_PATH` points to a saved file.

In [ ]:
EMB = {}          # (run name, seed, epoch) -> embeddings;  ("before", model) -> untrained embeddings
t0 = time.time()
clock = lambda: f"[{(time.time() - t0) / 60:.1f} min]"
run_name = lambda r: f"{short(r['model'])} · {r['objective']}"

if LOAD_PATH and os.path.exists(LOAD_PATH):
    data = np.load(LOAD_PATH, allow_pickle=False)
    assert np.array_equal(data["labels"], labels), "Saved file was made with different paintings"
    for key in data.files:
        if key == "labels":
            continue
        parts = key.split("|")
        EMB[("before", parts[1]) if parts[0] == "before" else (parts[0], int(parts[1]), int(parts[2]))] = data[key]
    print(f"Loaded {len(EMB)} embedding sets from {LOAD_PATH}")
else:
    for model_id in MODELS:
        AMP = USE_AMP and device == "cuda" and not any(k in model_id for k in AMP_OFF_FOR)
        print(f"\n=== {short(model_id)} (mixed precision: {AMP}) ===")
        base = load_backbone(model_id)
        EMB[("before", model_id)] = embed(base, images).astype(np.float16)
        for r in [r for r in RUNS if r["model"] == model_id]:
            LR, N_BLOCKS = r["lr"], r["n_blocks"]
            name = run_name(r)
            for seed in SEEDS:
                EMB[(name, seed, 0)] = EMB[("before", model_id)]
                def on_record(ep, m, h, loss):
                    if not np.isfinite(loss):
                        raise RuntimeError(f"{name}: loss is {loss} at epoch {ep}")
                    EMB[(name, seed, ep)] = embed(m, images).astype(np.float16)
                    if ep % 10 == 0:
                        print(f"  {name} s{seed} ep {ep:>2}: loss {loss:.3f} {clock()}")
                train_long(base, r["objective"], seed, on_record, warmup=WARMUPS[0])
                gc.collect(); torch.cuda.empty_cache()
        del base; gc.collect(); torch.cuda.empty_cache()

    save = {"labels": labels}
    for key, e in EMB.items():
        save["|".join(["before", key[1]] if key[0] == "before" else [key[0], str(key[1]), str(key[2])])] = e
    np.savez_compressed(SAVE_PATH, **save)
    print(f"\nSaved {len(EMB)} embedding sets to {SAVE_PATH} ({os.path.getsize(SAVE_PATH) / 1e6:.0f} MB) {clock()}")
    print("Download it from the Files panel (Colab) or the Output tab (Kaggle) to reuse it later.")

RUN_NAMES = list(dict.fromkeys(k[0] for k in EMB if k[0] != "before"))
EPOCHS_SAVED = sorted({k[2] for k in EMB if k[0] != "before"})
SEEDS_SAVED = sorted({k[1] for k in EMB if k[0] != "before"})
print("Runs:", RUN_NAMES, "| seeds:", SEEDS_SAVED, "| epochs:", EPOCHS_SAVED[0], "to", EPOCHS_SAVED[-1])

## Measures

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

def group_idx(names):
    return np.where(np.isin(labels, [class_names.index(n) for n in names if n in class_names]))[0]

def centroid_between(emb, ga, gb):
    e = unit_rows(emb)
    ca, cb = e[ga].mean(0), e[gb].mean(0)
    return 1 - ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb))

def pairwise_ratio(emb, ga, gb):
    """Average distance between cross-group pairs / average distance between same-group pairs."""
    e = unit_rows(emb)
    A, B = e[ga], e[gb]
    def within(X):
        n = len(X); S = X @ X.T
        return 1 - (S.sum() - n) / (n * (n - 1))
    return (1 - (A @ B.T).mean()) / ((within(A) + within(B)) / 2)

def knn_mixing(emb, ga, gb, k=K_NEIGHBOURS):
    """Share of nearest neighbours from the other group, relative to chance (0 = separate, 1 = fully mixed)."""
    idx = np.concatenate([ga, gb]); grp = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    e = unit_rows(emb[idx]); S = e @ e.T
    np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    other = (grp[nn] != grp[:, None]).mean(1)
    chance = np.where(grp == 0, len(gb), len(ga)) / (len(idx) - 1)
    return float((other / chance).mean())

def heldout_dprime(emb, ga, gb, folds=5):
    """Separation of the two groups along a linear boundary learned on other paintings (d′)."""
    X = unit_rows(emb[np.r_[ga, gb]]); y = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    vals = []
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=0).split(X, y):
        lda = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto").fit(X[tr], y[tr])
        p = lda.transform(X[te])[:, 0]
        a, b = p[y[te] == 0], p[y[te] == 1]
        vals.append(abs(a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2))
    return float(np.mean(vals))

MEASURES = {"centroid between": centroid_between, "pairwise ratio": pairwise_ratio,
            "neighbour mixing": knn_mixing, "held-out d′": heldout_dprime}
CLOSER_IF = {"centroid between": "down", "pairwise ratio": "down", "neighbour mixing": "up", "held-out d′": "down"}
# neighbour mixing can start at 0 (fully separate groups), so it is reported as an absolute change; the others as relative change
ABSOLUTE = {"neighbour mixing"}
TH = {"centroid between": 0.05, "pairwise ratio": 0.05, "held-out d′": 0.05, "neighbour mixing": 0.02}

def change(m, new, old):
    return new - old if m in ABSOLUTE else new / old - 1

def is_closer(m, c):
    return c < -TH[m] if CLOSER_IF[m] == "down" else c > TH[m]

def is_apart(m, c):
    return c > TH[m] if CLOSER_IF[m] == "down" else c < -TH[m]
GA, GB = group_idx(ABSTRACT), group_idx(REPRESENTATIONAL)
COL = {n: plt.get_cmap("tab10")(i) for i, n in enumerate(RUN_NAMES)}
LINE = {n: ("--" if "exposure" in n else "-") for n in RUN_NAMES}
print("Abstract:", ABSTRACT, f"({len(GA)} paintings) | Representational:", REPRESENTATIONAL, f"({len(GB)} paintings)")

## 1. Four measures of closeness over training

Original grouping. Change from before training (0 = unchanged): **relative** change for centroid between, pairwise ratio and d′ (−0.2 = 20% lower); **absolute** change for neighbour mixing, which can start at zero. Thin lines = seeds, thick = mean; solid = labels, dashed = exposure. For neighbour mixing *up* means closer; for the other three *down* means closer.

In [ ]:
TRACK = {}
for name in RUN_NAMES:
    for seed in SEEDS_SAVED:
        for ep in EPOCHS_SAVED:
            e = EMB[(name, seed, ep)].astype(np.float32)
            TRACK[(name, seed, ep)] = {m: f(e, GA, GB) for m, f in MEASURES.items()}

fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, m in zip(axes, MEASURES):
    for name in RUN_NAMES:
        mat = np.array([[change(m, TRACK[(name, s, ep)][m], TRACK[(name, s, 0)][m]) for ep in EPOCHS_SAVED] for s in SEEDS_SAVED])
        for line in mat:
            ax.plot(EPOCHS_SAVED, line, color=COL[name], ls=LINE[name], alpha=0.3, lw=1)
        ax.plot(EPOCHS_SAVED, mat.mean(0), color=COL[name], ls=LINE[name], lw=2.5, label=name)
    ax.axhline(0, color="k", ls=":", lw=1)
    kind = "absolute" if m in ABSOLUTE else "relative"
    ax.set_title(f"{m} ({kind} change)\ncloser = {CLOSER_IF[m]}", fontsize=10); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

rows = []
for name in RUN_NAMES:
    for s in SEEDS_SAVED:
        first, last = TRACK[(name, s, 0)], TRACK[(name, s, EPOCHS_SAVED[-1])]
        rows.append(dict(run=name, seed=s, **{m: change(m, last[m], first[m]) for m in MEASURES}))
final = pd.DataFrame(rows)
print("Change from before training to the final epoch (relative; neighbour mixing absolute):")
agg = final.groupby("run", sort=False)[list(MEASURES)].agg(["mean", "std"])
print((agg.xs("mean", axis=1, level=1).round(3).astype(str) + " ± " + agg.xs("std", axis=1, level=1).round(3).astype(str)).to_string())
print()
for name, g in final.groupby("run", sort=False):
    votes = []
    for m in MEASURES:
        closer = sum(is_closer(m, c) for c in g[m]); apart = sum(is_apart(m, c) for c in g[m])
        votes.append(f"{m}: closer {closer}/{len(g)}, apart {apart}/{len(g)}")
    print(f"{name}\n  " + "\n  ".join(votes))

## 2. Alternative group definitions and control splits

Final-epoch change for each grouping (mean across seeds; pairwise ratio relative, neighbour mixing absolute) and how many seeds moved **closer** on *both*. Rows starting with **control** compare groups that have nothing to do with abstraction: if exposure brings these closer too, it is merging styles in general.

In [ ]:
R_ = REPRESENTATIONAL
GROUPINGS = {
    "original": (ABSTRACT, R_),
    "Cubism counted as abstract": (ABSTRACT + ["Cubism"], R_),
    "Pop Art counted as representational": (ABSTRACT, R_ + ["Pop_Art"]),
    "Abstract Expressionism only": (["Abstract_Expressionism"], R_),
    "Colour Field only": (["Color_Field_Painting"], R_),
    "without Ukiyo-e": (ABSTRACT, [s for s in R_ if s != "Ukiyo_e"]),
    "without Impressionism": (ABSTRACT, [s for s in R_ if s != "Impressionism"]),
    "European figurative only (Baroque, Realism)": (ABSTRACT, ["Baroque", "Realism"]),
    "control: Baroque + Realism vs Impressionism + Ukiyo-e": (["Baroque", "Realism"], ["Impressionism", "Ukiyo_e"]),
    "control: Cubism vs Pop Art": (["Cubism"], ["Pop_Art"]),
    "control: Abstract Expressionism vs Colour Field": (["Abstract_Expressionism"], ["Color_Field_Painting"]),
}
last_ep = EPOCHS_SAVED[-1]
out = []
for gname, (a_names, b_names) in GROUPINGS.items():
    ga, gb = group_idx(a_names), group_idx(b_names)
    for name in RUN_NAMES:
        pr, km, both = [], [], 0
        for s in SEEDS_SAVED:
            e0, e1 = EMB[(name, s, 0)].astype(np.float32), EMB[(name, s, last_ep)].astype(np.float32)
            p = change("pairwise ratio", pairwise_ratio(e1, ga, gb), pairwise_ratio(e0, ga, gb))
            k = change("neighbour mixing", knn_mixing(e1, ga, gb), knn_mixing(e0, ga, gb))
            pr.append(p); km.append(k); both += int(is_closer("pairwise ratio", p) and is_closer("neighbour mixing", k))
        out.append(dict(grouping=gname, run=name, pairwise_ratio=np.mean(pr), neighbour_mixing=np.mean(km),
                        closer_seeds=f"{both}/{len(SEEDS_SAVED)}"))
table = pd.DataFrame(out)
for name in RUN_NAMES:
    print(f"\n{name}")
    print(table[table.run == name].drop(columns="run").set_index("grouping").round(3).to_string())

fig, axes = plt.subplots(1, len(RUN_NAMES), figsize=(8 * len(RUN_NAMES), 0.45 * len(GROUPINGS) + 1.8), squeeze=False)
for ax, name in zip(axes[0], RUN_NAMES):
    d = table[table.run == name]
    y = np.arange(len(d))
    colours = ["#999999" if g.startswith("control") else COL[name] for g in d.grouping]
    ax.barh(y, d.pairwise_ratio, color=colours)
    ax.set_yticks(y); ax.set_yticklabels(d.grouping, fontsize=8); ax.invert_yaxis()
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("change in pairwise ratio (negative = closer)"); ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

## 3. Every pair of styles

Change in the pairwise distance ratio between each pair of the 8 styles (final epoch vs before, mean across seeds). Blue = moved closer, red = moved apart. Pairs crossing the abstract–representational divide are outlined.

In [ ]:
Kc = len(class_names)
abstract_set, rep_set = set(ABSTRACT), set(REPRESENTATIONAL)
crosses = lambda i, j: ({class_names[i], class_names[j]} & abstract_set) and ({class_names[i], class_names[j]} & rep_set)
PAIR = {}
for name in RUN_NAMES:
    M = np.zeros((Kc, Kc))
    for s in SEEDS_SAVED:
        e0, e1 = EMB[(name, s, 0)].astype(np.float32), EMB[(name, s, last_ep)].astype(np.float32)
        for i in range(Kc):
            for j in range(i + 1, Kc):
                gi, gj = np.where(labels == i)[0], np.where(labels == j)[0]
                c = pairwise_ratio(e1, gi, gj) / pairwise_ratio(e0, gi, gj) - 1
                M[i, j] += c / len(SEEDS_SAVED); M[j, i] = M[i, j]
    PAIR[name] = M

v = max(np.abs(M).max() for M in PAIR.values())
fig, axes = plt.subplots(1, len(PAIR), figsize=(7 * len(PAIR), 6), squeeze=False, constrained_layout=True)
for ax, (name, M) in zip(axes[0], PAIR.items()):
    im = ax.imshow(np.where(np.eye(Kc, dtype=bool), np.nan, M), cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_xticks(range(Kc)); ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(Kc)); ax.set_yticklabels(class_names, fontsize=8)
    for i in range(Kc):
        for j in range(Kc):
            if i != j:
                ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=7)
                if crosses(i, j):
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.5))
    ax.set_title(name, fontsize=10)
fig.colorbar(im, ax=axes[0].tolist(), label="change in pairwise ratio (negative = closer)", shrink=0.8)
plt.show()

for name, M in PAIR.items():
    iu = [(i, j) for i in range(Kc) for j in range(i + 1, Kc)]
    cross = [M[i, j] for i, j in iu if crosses(i, j)]
    other = [M[i, j] for i, j in iu if not crosses(i, j)]
    print(f"{name}: abstract–representational pairs {np.mean(cross):+.3f} (n={len(cross)}), "
          f"all other pairs {np.mean(other):+.3f} (n={len(other)})")

## Verdict

In [ ]:
def informative(name, m):
    """Neighbour mixing can't register change if the groups never share neighbours (stuck at 0 before and after)."""
    if m not in ABSOLUTE:
        return True
    vals = [TRACK[(name, s, ep)][m] for s in SEEDS_SAVED for ep in (0, EPOCHS_SAVED[-1])]
    return max(vals) > TH[m]

for name in RUN_NAMES:
    g = final[final.run == name]; n = len(g)
    used = [m for m in MEASURES if informative(name, m)]
    closer_seeds = sum(all(is_closer(m, g.iloc[i][m]) for m in used) for i in range(n))
    apart_seeds = sum(all(is_apart(m, g.iloc[i][m]) for m in used) for i in range(n))
    t = table[table.run == name]
    alt = t[~t.grouping.str.startswith("control")]
    ctrl = t[t.grouping.str.startswith("control")]
    alt_closer = int(sum(is_closer("pairwise ratio", c) for c in alt.pairwise_ratio))
    ctrl_closer = int(sum(is_closer("pairwise ratio", c) for c in ctrl.pairwise_ratio))
    M = PAIR[name]
    iu = [(i, j) for i in range(Kc) for j in range(i + 1, Kc)]
    cross = np.mean([M[i, j] for i, j in iu if crosses(i, j)]); other = np.mean([M[i, j] for i, j in iu if not crosses(i, j)])
    skipped = [m for m in MEASURES if m not in used]
    print(f"\n{name}")
    print(f"  1. Measures used: {len(used)} of {len(MEASURES)}" + (f" (skipped {skipped}: stuck at zero)" if skipped else ""))
    print(f"     all say 'closer' in {closer_seeds} of {n} seeds; all say 'apart' in {apart_seeds} of {n} seeds")
    print(f"  2. Alternative groupings closer: {alt_closer} of {len(alt)}")
    print(f"  3. Control splits closer: {ctrl_closer} of {len(ctrl)}")
    print(f"  4. Abstract–representational pairs {cross:+.3f} vs all other pairs {other:+.3f}")
    specific = cross < other - TH["pairwise ratio"] and ctrl_closer <= len(ctrl) // 2
    if closer_seeds == n and alt_closer >= len(alt) - 1 and specific:
        print("  → ROBUST CONVERGENCE: the groups genuinely move closer, more than unrelated style pairs do.")
    elif closer_seeds == n and alt_closer >= len(alt) - 1:
        print("  → GENERAL CONVERGENCE: the groups move closer, but unrelated style pairs or control splits do too (styles merging overall).")
    elif apart_seeds == n:
        print("  → ROBUSTLY APART: every measure says the groups moved apart in every seed.")
    elif closer_seeds >= (n + 1) // 2:
        print("  → PARTIAL: most seeds say closer, but not all measures or groupings agree; treat as a weak effect.")
    else:
        print("  → NOT ROBUST: the measures disagree; the earlier result depended on how closeness was measured.")